## White Wine Quality App — Model Pipeline

### 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
%matplotlib inline

from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from scipy.stats import randint
import joblib

### 2. Load & Inspect Data

In [ ]:
wp = pd.read_csv(Path('white-wine-price-rating.csv'))
print('Shape:', wp.shape)
print()
wp.info()

In [ ]:
wp.head(5)

### 3. Data Cleaning

In [ ]:
# Strip whitespace from Year and drop non-vintage rows
wp['Year'] = wp['Year'].str.strip()
wp = wp[wp['Year'] != 'N.V.'].copy()
wp['Year'] = wp['Year'].astype(int)

# Drop rows with missing Region / RegionalVariety
wp = wp.dropna(subset=['Region', 'RegionalVariety']).copy()

print('Clean shape:', wp.shape)
print('Year range:', wp['Year'].min(), '-', wp['Year'].max())

### 4. Create Quality Label
`WineRating >= 4.3` = Good (1), otherwise Bad (0)

In [ ]:
wp['quality'] = (wp['WineRating'] >= 4.3).astype(int)

print('Quality distribution:')
print(wp['quality'].value_counts())
print(f"\nGood ratio: {wp['quality'].mean():.1%}")

sns.countplot(wp, x='quality')
plt.xticks([0, 1], ['Bad (0)', 'Good (1)'])
plt.title('Quality Label Distribution')
plt.show()

### 5. Exploratory Data Analysis

In [ ]:
# Price vs Quality
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

wp.boxplot(column='WinePrice', by='quality', ax=axes[0])
axes[0].set_title('Wine Price vs Quality')
axes[0].set_xlabel('Quality')
axes[0].set_ylabel('Price')

sns.barplot(x='quality', y='Year', data=wp, ax=axes[1])
axes[1].set_title('Vintage Year vs Quality')
axes[1].set_xticklabels(['Bad (0)', 'Good (1)'])

plt.tight_layout()
plt.show()

In [ ]:
# Good wine rate by region (min 20 samples)
region_quality = wp.groupby('Region')['quality'].agg(['mean', 'count'])
region_quality = region_quality[region_quality['count'] >= 20].sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
region_quality['mean'].plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Good Wine Rate by Region (min 20 samples)')
ax.set_ylabel('Proportion Rated Good')
ax.set_xlabel('Region')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Good wine rate by grape variety (min 20 samples)
variety_quality = wp.groupby('RegionalVariety')['quality'].agg(['mean', 'count'])
variety_quality = variety_quality[variety_quality['count'] >= 20].sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
variety_quality['mean'].plot(kind='bar', ax=ax, color='coral')
ax.set_title('Good Wine Rate by Variety (min 20 samples)')
ax.set_ylabel('Proportion Rated Good')
ax.set_xlabel('Variety')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 6. Feature Engineering

In [ ]:
# Select consumer-facing features
features = ['Year', 'Region', 'RegionalVariety', 'WinePrice']
target = 'quality'

df = wp[features + [target]].copy()

# Encode categorical columns
le_region = LabelEncoder()
le_variety = LabelEncoder()

df['Region'] = le_region.fit_transform(df['Region'])
df['RegionalVariety'] = le_variety.fit_transform(df['RegionalVariety'])

print('Feature sample:')
print(df.head())
print()
print('Regions available:', list(le_region.classes_))
print()
print('Varieties available:', list(le_variety.classes_))

### 7. Train / Test Split & Scaling

In [ ]:
X = df[features].values
y = df[target].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print('Train size:', X_train.shape)
print('Test size: ', X_test.shape)
print('Good rate in train:', y_train.mean().round(3))
print('Good rate in test: ', y_test.mean().round(3))

### 8. Baseline Random Forest

In [ ]:
RF = RandomForestClassifier(n_estimators=200, random_state=1, class_weight='balanced')
RF.fit(X_train, y_train)
predictions = RF.predict(X_test)

print('Accuracy: ', round(accuracy_score(y_test, predictions), 5))
print('ROC-AUC: ', round(roc_auc_score(y_test, RF.predict_proba(X_test)[:,1]), 5))
print()
print(classification_report(y_test, predictions, target_names=['Bad (0)', 'Good (1)']))

### 9. Cross Validation

In [ ]:
X_full = scaler.fit_transform(df[features].values)
cv_scores = cross_val_score(RF, X_full, y, cv=10)
print(f'CV Mean: {cv_scores.mean():.5f} (+/- {cv_scores.std():.5f})')

### 10. Hyperparameter Tuning — Randomized Search

In [ ]:
param_dist = {
    'n_estimators'     : randint(100, 500),
    'max_depth'        : [None, 5, 10, 15, 20],
    'min_samples_split': randint(2, 10),
    'min_samples_leaf' : randint(1, 5),
}

RF_random = RandomizedSearchCV(
    RandomForestClassifier(random_state=1, class_weight='balanced'),
    param_distributions=param_dist,
    n_iter=50, scoring='roc_auc', cv=10, random_state=1, n_jobs=-1
)
RF_random.fit(X_train, y_train)
print('Best params:', RF_random.best_params_)

In [ ]:
best_RF = RF_random.best_estimator_
predictions = best_RF.predict(X_test)

print('Tuned RF Accuracy: ', round(accuracy_score(y_test, predictions), 5))
print('Tuned RF ROC-AUC: ', round(roc_auc_score(y_test, best_RF.predict_proba(X_test)[:,1]), 5))
print()
print(classification_report(y_test, predictions, target_names=['Bad (0)', 'Good (1)']))

### 11. Feature Importance

In [ ]:
importances = pd.Series(best_RF.feature_importances_, index=features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importances — Tuned Random Forest')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print()
for feat, score in importances.sort_values(ascending=False).items():
    print(f'{score:.4f}  {feat}')

### 12. Save Model Artefacts
Save the trained model, scaler and label encoders so the Streamlit app can load them.

In [ ]:
joblib.dump(best_RF,    'model.pkl')
joblib.dump(scaler,     'scaler.pkl')
joblib.dump(le_region,  'le_region.pkl')
joblib.dump(le_variety, 'le_variety.pkl')

print('Saved: model.pkl, scaler.pkl, le_region.pkl, le_variety.pkl')